---
title: "Features"
jupyter: python3
---

Let's see some of **Specification Curve**'s features in action.

In [ ]:
# | echo: false

import matplotlib_inline.backend_inline

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

## Basic Use

Here's an example of using **Specification Curve**. You say which outcome (`y`) and which regressor of interest (`x`) you care about, and which choices should vary. Every argument accepts a single value or a list of alternatives, and every term is written in [pyfixest](https://pyfixest.org/) formula syntax, so transformations such as `log(c1)` work anywhere.

In the example below, each of the two controls is switched in and out, and the model is run without fixed effects, with fixed effects for `group1`, and with fixed effects for both groups.

In [ ]:
import specification_curve as sc

df = sc.load_example_data1()
sco = sc.SpecificationCurve(
    df,
    y="y1",
    x="x1",
    controls=["c1", "c2"],
    fe=[None, "group1", "group1 + group2"],
)
sco.fit()
sco.plot();

Each column is one specification. The panels under the curve show which choices it makes; markers are coloured blue when the coefficient is positive and significant (at the 0.05 level), red when it is negative and significant, and black otherwise.

You can also describe the specifications with a pyfixest formula. Several outcomes, `sw()` and `sw0()` work as they do in pyfixest, and `comb()` / `comb0()` stand for every combination of their terms (with `comb0()` including none of them). The `x` argument names the coefficient to plot.

Here's an example, for which we first generate some data.

In [ ]:
import numpy as np
import pandas as pd

# Set seed for random numbers
seed_for_prng = 78557
# prng=probabilistic random number generator
prng = np.random.default_rng(seed_for_prng)

# generate some fake data
n_samples = 1000
x_2 = prng.integers(2, size=n_samples)
x_1 = prng.random(size=n_samples)
x_3 = prng.integers(3, size=n_samples)
x_4 = prng.random(size=n_samples)
x_5 = x_1 + 0.05 * np.random.randn(n_samples)
x_beta = -1 - 3.5 * x_1 + 0.2 * x_2 + 0.3 * x_3  # NB: coefficient is -3.5
prob = 1 / (1 + np.exp(-x_beta))
y = prng.binomial(n=1, p=prob, size=n_samples)
y2 = prng.binomial(n=1, p=prob * 0.98, size=n_samples)
df_logit = pd.DataFrame(
    [x_1, x_2, x_3, x_4, x_5, y, y2], ["x_1", "x_2", "x_3", "x_4", "x_5", "y", "y2"]
).T

In [ ]:
sc_form = sc.SpecificationCurve(
    df_logit, "y + y2 ~ sw(x_1, x_5) + x_2 + comb0(x_3, x_4)", x=["x_1", "x_5"]
)
sc_form.fit()
sc_form.plot();

Note that "x_2" is included in all specifications.

## Retrieving estimates

The results are a tidy data frame with one row per specification:

In [ ]:
sco.results.head()

## Inference under the null

Although not everyone is convinced, it may be informative to run some statistical inferences on the specification curve. These ask: considering the full set of reasonable specifications jointly, how inconsistent are the results with the null hypothesis of no effect? You can find more details in [@simonsohn2020specification]. 

In [ ]:
dfn = sc.load_example_data3()
dfn.head()

As normal, we create a specification curve object and fit it.

In [ ]:
sco = sc.SpecificationCurve(dfn, y=["y1", "y2"], x="x1", controls=["c1", "c2", "c3"])
sco.fit()

Now we fit the curve under the null with `.fit_null()`. By default, for each specification the estimated effect is removed from the outcome, the rows are resampled, and every specification is re-estimated. This is available for linear models (`model="feols"`). For other models, and for experiments, `method="permutation"` shuffles the variable of interest between rows instead; that is only valid if it is as good as randomly assigned. The `cluster` and `strata` arguments resample whole clusters, or within strata.

You should really use a large number of resamples (eg 500) but be warned it takes a long time to run this many; `n_jobs` runs them in parallel.

In [ ]:
sco.fit_null(n_boot=10, seed=0)

And now we can plot it to see what the likely range of the coefficient would be under the null:

In [ ]:
sco.plot(show_null_curve=True);

Finally, there is a summary of three tests of the curve against the null, following [@simonsohn2020specification]. Each asks whether a feature of the observed curve is more extreme than you would expect by chance: the median estimate, the share of estimates that are significant, and Stouffer's Z, which combines the evidence of all the specifications.

In [ ]:
sco.null_summary

And, once you have done all of this fitting, you can examine it all simply by calling the object:

In [ ]:
sco

## Saving results to file

Save the plot to file (the format is inferred from file extension):

```python
sco.plot(save_path="test_fig.pdf")
```

## Running on subsets of the data

You can run every specification on several samples with the `subsets` argument, which maps a name for each sample to a query. The helper `subsets_by` builds one sample per level of a column.

In [ ]:
sco = sc.SpecificationCurve(
    df,
    y="y1",
    x="x1",
    controls=["c1", "c2"],
    subsets=sc.subsets_by(df, "group2", include_all=True),
)
sco.fit()
sco.plot();

## Using multiple regressors of interest

Sometimes, you'd like to check different independent variables (and the coefficients they come with following a regression). This is achieved by passing a list to the `x` argument of `SpecificationCurve`. These variations on the independent variable are labelled by `x` in the plot.

In [ ]:
df = sc.load_example_data1()
sco = sc.SpecificationCurve(df, y="y1", x=["x1", "x2"], controls=["c1", "c2"])
sco.fit()
sco.plot();

## Excluding some combinations of controls

Some controls may be redundant, and you might want to exclude them both being used together. The `exclude` keyword argument achieves this.

In the below example, `"c1"` and `"c2"` are never run in the same specification.

In [ ]:
sco = sc.SpecificationCurve(
    df, y="y1", x="x1", controls=["c1", "c2", "x2"], exclude=[["c1", "c2"]]
)
sco.fit()
sco.plot();

## Always include some controls in all specifications

Likewise, there will be times when you always wish to include a particular control in specifications. The `always` keyword argument helps you to achieve this.

In the example below, we ask that `"c1"` is included in every specification.

In [ ]:
sco = sc.SpecificationCurve(df, y="y1", x="x1", controls=["c2", "x2"], always="c1")
sco.fit()
sco.plot();

## Flexing the style for very large numbers of specifications

The default plot type isn't suitable for very large numbers of specifications, but it does automatically switch to a style suited to a large number of specifications.

Here's an example

In [ ]:
# Generate some fake data

n_samples = 2000
# Number of dimensions of continuous
n_dim = 8
c_rnd_vars = prng.random(size=(n_dim, n_samples))
c_rnd_vars_names = [f"c_{i}" for i in range(np.shape(c_rnd_vars)[0])]
y_1 = (
    0.4 * c_rnd_vars[0, :]  # This is the true value of the coefficient
    - 0.2 * c_rnd_vars[1, :]
    + 0.3 * prng.standard_normal(n_samples)
)
# Next line causes y_2 ests to be much more noisy
y_2 = y_1 - 0.3 * np.abs(prng.standard_normal(n_samples))
df = pd.DataFrame([y_1, y_2], ["y1", "y2"]).T
for i, col_name in enumerate(c_rnd_vars_names):
    df[col_name] = c_rnd_vars[i, :]

controls = c_rnd_vars_names[1:]

# Run it with Specification Curve
sco = sc.SpecificationCurve(
    df, y=["y1", "y2"], x=c_rnd_vars_names[0], controls=controls
)
sco.fit()
sco.plot();

## Flagging a preferred specification

Often, in practice, you will have a *preferred* specification that you will use as your estimate. You can have it flagged with the `highlight` keyword argument, which takes the values that identify the specification (or its `spec_id`, or a query string).

In the example below, the preferred specification comes out as being close to the known answer that we constructed.

In [ ]:
sco.plot(highlight={"y": "y1", "controls": controls});

## Customising the chart

`plot` returns the matplotlib figure and a dictionary of its axes, so you can adjust anything afterwards. It also takes options for the most common changes: `sort_by` orders the specifications, `color_by` colours them by a choice rather than by significance, `panels` picks which choices are shown underneath, and `labels` renames things for display.

In [ ]:
fig, axes = sco.plot(
    sort_by="y",
    color_by="y",
    panels=["y"],
    labels={"y": "Outcome", "y1": "First outcome", "y2": "Second outcome"},
)
axes["curve"].set_title("Sorted and coloured by outcome");

## Using models other than Ordinary Least Squares

The default model is OLS (pyfixest's `feols`), but you can also use Poisson regression (`"fepois"`), generalised linear models (`"feglm"`, with a `family`) and quantile regression (`"quantreg"`).

In [ ]:
sco = sc.SpecificationCurve(
    df_logit,
    y=["y", "y2"],
    x=["x_1", "x_5"],
    controls=["x_2", "x_3", "x_4"],
    model="feglm",
    family="logit",  # "probit" also works
)
sco.fit()
sco.plot();